In [0]:
from pyspark.sql.functions import (
    add_months,
    col,
    countDistinct,
    date_add,
    datediff,
    lit,
    max,
    round,
    sum,
    when
)

In [0]:
silver_df=spark.table(
    "workspace.default.silver_sales_v2"
)


In [0]:
required_columns = [
    "CustomerID",
    "InvoiceNo",
    "InvoiceDate",
    "Quantity",
    "UnitPrice"
]

missing_columns = [
    column
    for column in required_columns
    if column not in silver_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if silver_df.limit(1).count() == 0:
    raise ValueError("Silver sales table is empty")

In [0]:

valid_sales_df=silver_df.filter(
    (col("CustomerID")!="-1")& (col("Quantity")>0)&(col("UnitPrice")>0)
)
if valid_sales_df.limit(1).count() == 0:
    raise ValueError(
        "No valid sales remain after applying RFM business rules"
    )

sales_with_revenue_df=valid_sales_df.withColumn("Revenue",col("Quantity")*col("UnitPrice"))



In [0]:


latest_date_df = sales_with_revenue_df.agg(
    max("InvoiceDate").alias("LatestDate")
)

display(latest_date_df)
reference_date_df = latest_date_df.select(
    date_add(col("LatestDate"), 1).alias("ReferenceDate")
)



LatestDate
2026-08-26T21:59:00.000Z


In [0]:
reference_date = reference_date_df.first()["ReferenceDate"]

In [0]:


analysis_sales_df = sales_with_revenue_df.filter(
    col("InvoiceDate") >= add_months(lit(reference_date), -12)
)



In [0]:
customer_rfm_df = analysis_sales_df.groupBy("CustomerID").agg(
    max("InvoiceDate").alias("LastPurchaseDate"),
    countDistinct("InvoiceNo").alias("Frequency"),
    round(
    sum("Revenue"),
    2
).alias("Monetary")
)



In [0]:


rfm_df = customer_rfm_df.withColumn(
    "Recency",
    datediff(
        lit(reference_date),
        col("LastPurchaseDate")
    )
)



In [0]:


rfm_scored_df = rfm_df.withColumn(
    "R_Score",
    when(col("Recency")<=7,5)
    .when(col("Recency")<=14,4)
   .when(col("Recency")<=30,3)
    .when(col("Recency")<=60,2)
    .otherwise(1)
)


In [0]:
rfm_scored_df = rfm_scored_df.withColumn(
    "F_Score",
   when(col("Frequency")>=4,5)
   .when(col("Frequency")>=3,4)
   .when(col("Frequency")>=2,3)
  
   .otherwise(1)
)


In [0]:
monetary_boundaries = rfm_df.approxQuantile(
    "Monetary",
    [0.2, 0.4, 0.6, 0.8],
    0.0
)



In [0]:
q20, q40, q60, q80 = monetary_boundaries
rfm_scored_df = rfm_scored_df.withColumn(
    "M_Score",
    when(col("Monetary") <= q20, 1)
    .when(col("Monetary") <= q40, 2)
    .when(col("Monetary") <= q60, 3)
    .when(col("Monetary") <= q80, 4)
    .otherwise(5)
    
)



In [0]:
rfm_segmented_df = rfm_scored_df.withColumn(
    "Segment",
    when(
        (col("R_Score") >= 4) &
        (col("F_Score") >= 4) &
        (col("M_Score") >= 4),
        "Champions"
    )
    .when(
    (col("R_Score") >= 3) &
    (col("F_Score") >= 3),
    "Loyal Customers"
)
    .when(
    (col("R_Score") >= 4) &
    (col("F_Score") <= 2),
    "Potential Loyalists"
)
    .when(
    (col("R_Score") <= 2) &
    (
        (col("F_Score") >= 3) |
        (col("M_Score") >= 4)
    ),
    "At Risk")

    .when(
    (col("R_Score") <= 2) &
    (col("F_Score") <= 2) &
    (col("M_Score") <= 3),
    "Hibernating"
)

    .otherwise("Needs Attention")
)


In [0]:
rfm_segmented_df = (
    rfm_segmented_df
    .withColumn(
        "ReferenceDate",
        lit(reference_date)
    )
    .select(
        "CustomerID",
        "ReferenceDate",
        "LastPurchaseDate",
        "Recency",
        "Frequency",
        "Monetary",
        "R_Score",
        "F_Score",
        "M_Score",
        "Segment"
    )
)

In [0]:
segment_summary_df = rfm_segmented_df.groupBy("Segment").count()

display(segment_summary_df)

Segment,count
Loyal Customers,6
Needs Attention,11
Potential Loyalists,10
Champions,1


Databricks visualization. Run in Databricks to view.

In [0]:
rfm_segmented_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_customer_rfm")

In [0]:
saved_rfm_df = spark.table(
    "workspace.default.gold_customer_rfm"
)

print(saved_rfm_df.count())
display(saved_rfm_df)

28


CustomerID,ReferenceDate,LastPurchaseDate,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment
13047,2026-08-27,2026-08-10T10:28:00.000Z,17,2,24.66,3,3,4,Loyal Customers
14688,2026-08-27,2026-08-10T10:02:00.000Z,17,1,15.30,3,1,2,Needs Attention
18102,2026-08-27,2026-08-26T21:43:00.000Z,1,2,39.79,5,3,5,Loyal Customers
14005,2026-08-27,2026-08-12T10:20:00.000Z,15,1,17.50,3,1,2,Needs Attention
12583,2026-08-27,2026-08-10T10:41:00.000Z,17,2,27.55,3,3,4,Loyal Customers
14007,2026-08-27,2026-08-12T10:30:00.000Z,15,1,29.00,3,1,4,Needs Attention
17853,2026-08-27,2026-08-26T21:56:00.000Z,1,1,13.56,5,1,2,Potential Loyalists
17450,2026-08-27,2026-08-11T10:44:00.000Z,16,1,14.56,3,1,2,Needs Attention
12662,2026-08-27,2026-08-11T09:35:00.000Z,16,1,10.20,3,1,1,Needs Attention
18202,2026-08-27,2026-08-26T21:58:00.000Z,1,1,19.00,5,1,3,Potential Loyalists


In [0]:
segment_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_rfm_segment_summary")

In [0]:
saved_summary_df = spark.table(
    "workspace.default.gold_rfm_segment_summary"
)

display(saved_summary_df)

Segment,count
Champions,1
Loyal Customers,6
Potential Loyalists,10
Needs Attention,11
